# mesh_gpu — run the GPU exercises on Google Colab (no Docker)

This notebook compiles the **existing** sources and runs them on Colab's free GPU:
the CUDA programs with a direct `nvcc` call, and the Kokkos programs (CUDA backend)
via the project's CMake. The project source is not modified — the same files build
here and inside the Docker image.

First set a GPU runtime: **Runtime -> Change runtime type -> Hardware accelerator: GPU** (the free tier gives an NVIDIA **T4** = `sm_75`).

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the project source (with the meshes)
Clone the repo; the meshes under `data/` come with it. If the repo is **private**,
either put a token in the URL, or skip the clone and upload `source/` and `data/`
with the file browser on the left.

In [ ]:
REPO_URL = "https://github.com/julesdichamp/mesh_gpu.git"
!git clone --depth 1 $REPO_URL mesh_gpu
%cd mesh_gpu
!ls data

## 3. Build the dependencies
libMeshb (built from source into `/content/local`) and the header-only cxxopts.
Thrust ships with the CUDA toolkit, so nothing to install for it.

In [ ]:
%%bash
set -e
git clone --depth 1 https://github.com/LoicMarechal/libMeshb /content/libMeshb
cmake -S /content/libMeshb -B /content/libMeshb/build -DCMAKE_BUILD_TYPE=Release \
      -DCMAKE_INSTALL_PREFIX=/content/local > /tmp/libmeshb_build.log 2>&1
cmake --build /content/libMeshb/build -j2 >> /tmp/libmeshb_build.log 2>&1
cmake --install /content/libMeshb/build >> /tmp/libmeshb_build.log 2>&1
echo 'libMeshb installed to /content/local'

mkdir -p /content/include
curl -sL https://raw.githubusercontent.com/jarro2783/cxxopts/v3.2.0/include/cxxopts.hpp \
     -o /content/include/cxxopts.hpp
echo 'cxxopts header ready'

## 4. Compile the existing CUDA sources
The free GPU is a T4 (`sm_75`). If `nvidia-smi` above shows a different card,
change `-arch` (V100=70, A100=80, L4=89) or use `-arch=native`.

In [ ]:
%%bash
set -e
for f in area_triangles area_median_cells; do
  nvcc -O2 -std=c++17 -arch=sm_75 \
    -I source -I /content/include -I /content/local/include \
    source/cuda/$f.cu -L /content/local/lib -lMeshb.8 -o ${f}_cuda
  echo "built ${f}_cuda"
done

## 5. Run the CUDA programs
Each prints its total; the two totals should match. CSVs are written to `output/`.

In [ ]:
%%bash
./area_triangles_cuda      data/unit_square_132.meshb
./area_median_cells_cuda   data/unit_square_132.meshb --method geometric
./area_median_cells_cuda   data/unit_square_132.meshb --method third
echo '--- CSV heads ---'
head -3 output/triangle_areas.csv
head -3 output/median_cell_areas.csv

## 6. Kokkos on the GPU (CUDA backend)
Kokkos + CUDA is best built through the project's own CMake, which handles the
compile flags. First build **Kokkos** with the CUDA backend (T4 = `TURING75`) and
install **cxxopts** as a CMake package (the project uses `find_package(cxxopts)`).
This cell takes a few minutes (it compiles Kokkos' device code).

In [ ]:
%%bash
set -e
# Kokkos with CUDA (arch for the T4; change if nvidia-smi shows another card:
# VOLTA70, AMPERE80, ADA89, HOPPER90). -c advice.detachedHead=false just quiets
# git's normal "detached HEAD" note when checking out a tag.
git -c advice.detachedHead=false clone --depth 1 --branch 4.7.04 https://github.com/kokkos/kokkos /content/kokkos-src
cmake -S /content/kokkos-src -B /content/kokkos-src/build -DCMAKE_BUILD_TYPE=Release \
      -DCMAKE_INSTALL_PREFIX=/content/kokkos -DCMAKE_CXX_STANDARD=17 \
      -DKokkos_ENABLE_SERIAL=ON -DKokkos_ENABLE_CUDA=ON -DKokkos_ARCH_TURING75=ON \
      > /tmp/kokkos_build.log 2>&1
cmake --build /content/kokkos-src/build -j2 >> /tmp/kokkos_build.log 2>&1
cmake --install /content/kokkos-src/build >> /tmp/kokkos_build.log 2>&1
echo 'Kokkos (CUDA) installed to /content/kokkos'

# cxxopts as a CMake package (installed next to libMeshb in /content/local).
git -c advice.detachedHead=false clone --depth 1 --branch v3.2.0 https://github.com/jarro2783/cxxopts /content/cxxopts-src
cmake -S /content/cxxopts-src -B /content/cxxopts-src/build \
      -DCMAKE_INSTALL_PREFIX=/content/local \
      -DCXXOPTS_BUILD_EXAMPLES=OFF -DCXXOPTS_BUILD_TESTS=OFF > /tmp/cxxopts_build.log 2>&1
cmake --install /content/cxxopts-src/build >> /tmp/cxxopts_build.log 2>&1
echo 'cxxopts (CMake package) installed to /content/local'

## 7. Build & run the Kokkos programs
Configure the project with CMake (pointing it at libMeshb/cxxopts in `/content/local`
and Kokkos in `/content/kokkos`) and build only the `*_kokkos` targets, then run them.

In [ ]:
%%bash
set -e
cmake -S source -B build -DCMAKE_PREFIX_PATH="/content/local;/content/kokkos" \
      -DCMAKE_CUDA_ARCHITECTURES=75 > /tmp/cmake_cfg.log 2>&1
cmake --build build -j2 --target area_triangles_kokkos area_median_cells_kokkos \
      > /tmp/cmake_build.log 2>&1
echo 'built the Kokkos programs (CUDA backend)'

./build/area_triangles_kokkos      data/unit_square_132.meshb
./build/area_median_cells_kokkos   data/unit_square_132.meshb --method geometric
./build/area_median_cells_kokkos   data/unit_square_132.meshb --method third